# UR3e + Hand-E Manipulation (`UR3Pick`)

Load the UR3 pick model, **visually verify it works**, run a short **training test**,
roll out the policy, and save an artifact the real-robot loop can load.

Headless-Linux friendly: trains on **GPU if an NVIDIA device is present, otherwise CPU**.
Mirror of `manipulation UR10.ipynb`, adapted for `UR3Pick` (21D obs, 7D action).

### 1 · Backend setup (GPU if available, else CPU)

In [ ]:
# @title Detect backend and configure MuJoCo / JAX
import os
import platform
import subprocess


def _nvidia_available() -> bool:
    """True if nvidia-smi is callable (Linux + NVIDIA GPU)."""
    try:
        return subprocess.run(
            ["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.PIPE
        ).returncode == 0
    except FileNotFoundError:
        return False


system = platform.system()

if system == "Darwin":  # macOS - CPU only, windowed GL
    os.environ["MUJOCO_GL"] = "glfw"
    os.environ["JAX_PLATFORM_NAME"] = "cpu"
    print("macOS detected: MUJOCO_GL=glfw, JAX on CPU")
elif _nvidia_available():  # Linux + NVIDIA GPU (lab machine / cluster)
    os.environ["MUJOCO_GL"] = "egl"  # headless GPU rendering
    os.environ["XLA_FLAGS"] = (
        os.environ.get("XLA_FLAGS", "") + " --xla_gpu_triton_gemm_any=True"
    )
    print("NVIDIA GPU detected: MUJOCO_GL=egl, JAX on GPU")
else:  # Linux without a GPU
    os.environ["JAX_PLATFORM_NAME"] = "cpu"
    os.environ["MUJOCO_GL"] = "egl"  # NOTE: osmesa is NOT installed on the lab box
    print("No NVIDIA GPU: JAX on CPU. Rendering uses EGL - if it fails, "
          "rendering needs a GPU/EGL or `apt install libosmesa6` + MUJOCO_GL=osmesa.")

# Smoke-test MuJoCo + rendering backend.
try:
    import mujoco
    mujoco.MjModel.from_xml_string("<mujoco/>")
    print("MuJoCo OK, MUJOCO_GL =", os.environ["MUJOCO_GL"])
except Exception as e:
    print("MuJoCo backend test failed:", e)

### 2 · Imports & version check

In [ ]:
import jax, mujoco, brax, flax
print("JAX  :", jax.__version__)
print("MuJoCo:", mujoco.__version__)
print("Brax :", brax.__version__)
print("Flax :", flax.__version__)
print("JAX devices:", jax.devices())

In [ ]:
# @title Imports
from datetime import datetime
import functools
import json
from pathlib import Path

from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from flax import serialization
import jax.numpy as jp
import matplotlib.pyplot as plt
import mediapy as media
import numpy as np

from mujoco import mjx
from mujoco_playground import registry, wrapper
from mujoco_playground.config import manipulation_params

np.set_printoptions(precision=3, suppress=True, linewidth=100)
print("Manipulation envs:", registry.manipulation.ALL_ENVS)

### 3 · Load the `UR3Pick` model - visual check

Render the `low_home` keyframe to confirm the UR3 + Hand-E + box load cleanly
(no floor penetration, gripper attached). This is the "does it work?" check.

In [ ]:
ENV_NAME = "UR3Pick"
XML_PATH = "../../mujoco_playground/_src/manipulation/my_ur3/xmls/mjx_single_cube_position_ur3.xml"

env = registry.load(ENV_NAME)
env_cfg = registry.get_default_config(ENV_NAME)
print(f"action_size = {env.action_size}  (expect 7: 6 arm + 1 Hand-E)")
print(f"observation_size = {env.observation_size}  (expect 21)")

mj_model = mujoco.MjModel.from_xml_path(XML_PATH)
mj_data = mujoco.MjData(mj_model)
print(f"nu={mj_model.nu}  nq={mj_model.nq}  nkey={mj_model.nkey}  "
      f"nsensor={mj_model.nsensor}")

In [ ]:
# Render the low_home keyframe (static visual check)
key_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_KEY, "low_home")
mujoco.mj_resetDataKeyframe(mj_model, mj_data, key_id)
mujoco.mj_forward(mj_model, mj_data)

renderer = mujoco.Renderer(mj_model, height=480, width=640)
cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_FREE
cam.lookat[:] = [0.3, 0.0, 0.15]   # UR3 workspace is small / close to base
cam.distance = 1.2
cam.azimuth = 130
cam.elevation = -20

renderer.update_scene(mj_data, camera=cam)
media.show_image(renderer.render())

### 4 · Training config

In [ ]:
ppo_params = manipulation_params.brax_ppo_config(ENV_NAME)
print("Default UR3Pick PPO params:")
for k, v in ppo_params.items():
    print(f"  {k}: {v}")

In [ ]:
from copy import deepcopy

# -- Quick TEST run. Bump num_timesteps to 20_000_000 for a real policy. --
test_params = deepcopy(ppo_params)
test_params["num_timesteps"] = 2_000_000        # short smoke; full default is 20M

on_gpu = jax.devices()[0].platform == "gpu"
if not on_gpu:
    # CPU is much slower - shrink so the test finishes in minutes.
    test_params["num_timesteps"] = 300_000
    test_params["num_envs"] = 256
    test_params["batch_size"] = 128
    test_params["num_evals"] = 2
    print("Running on CPU - reduced to a tiny smoke config.")

ppo_params = test_params
print(f"num_timesteps={ppo_params['num_timesteps']}, "
      f"num_envs={ppo_params['num_envs']}, on_gpu={on_gpu}")

### 5 · Train (PPO)

In [ ]:
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]


def progress(num_steps, metrics):
    times.append(datetime.now())
    x_data.append(num_steps)
    y_data.append(metrics["eval/episode_reward"])
    y_dataerr.append(metrics["eval/episode_reward_std"])
    # print-based so it also works in a headless/background run
    print(f"[{num_steps:>10d} steps]  reward = "
          f"{metrics['eval/episode_reward']:.3f} "
          f"+/- {metrics['eval/episode_reward_std']:.3f}", flush=True)


ppo_training_params = dict(ppo_params)
network_factory = ppo_networks.make_ppo_networks
nf_params = {}
if "network_factory" in ppo_params:
    nf_params = dict(ppo_params["network_factory"])
    del ppo_training_params["network_factory"]
    network_factory = functools.partial(
        ppo_networks.make_ppo_networks, **nf_params
    )

train_fn = functools.partial(
    ppo.train,
    **ppo_training_params,
    network_factory=network_factory,
    progress_fn=progress,
    seed=1,
)

In [ ]:
make_inference_fn, params, metrics = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
)
best_idx = int(np.argmax(y_data)) if y_data else 0
print(f"\ntime to jit  : {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")
if y_data:
    print(f"best reward  : {y_data[best_idx]:.3f} +/- {y_dataerr[best_idx]:.3f} "
          f"at step {x_data[best_idx]}")

In [ ]:
# Training curve
if y_data:
    plt.figure(figsize=(7, 4))
    plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")
    plt.xlabel("# environment steps")
    plt.ylabel("reward per episode")
    plt.title(f"UR3Pick training (final y={y_data[-1]:.3f})")
    plt.show()

### 6 · Visualize a rollout

In [ ]:
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
jit_inference_fn = jax.jit(make_inference_fn(params, deterministic=True))

In [ ]:
rng = jax.random.PRNGKey(0)
state = jit_reset(rng)

# Reuse the same MuJoCo model for replay.
roll_data = mujoco.MjData(mj_model)
roll_renderer = mujoco.Renderer(mj_model, height=480, width=640)

num_steps = 200
render_every = 2
frames = []

for t in range(num_steps):
    action, _ = jit_inference_fn(state.obs, rng)
    state = jit_step(state, action)

    roll_data.qpos[:] = np.array(state.data.qpos)
    roll_data.qvel[:] = np.array(state.data.qvel)
    roll_data.ctrl[:] = np.array(state.data.ctrl)
    if hasattr(state.data, "mocap_pos"):
        roll_data.mocap_pos[:] = np.array(state.data.mocap_pos)
    if hasattr(state.data, "mocap_quat"):
        roll_data.mocap_quat[:] = np.array(state.data.mocap_quat)
    mujoco.mj_forward(mj_model, roll_data)

    if t % render_every == 0:
        roll_renderer.update_scene(roll_data, camera=cam)
        frames.append(roll_renderer.render())

print(f"rendered {len(frames)} frames")
media.show_video(frames, fps=1.0 / env_cfg.ctrl_dt / render_every)

### 7 · Save the policy (loadable by `ur3_realrobot_pickloop.py`)

Writes `params.msgpack` + `metadata.json` with the exact keys
`ur3_realrobot_dependencies.load_policy_fn` reads (`env_name`, `obs_dim`,
`action_dim`, `network_factory`).

In [ ]:
save_dir = Path("../../evaluation/downloaded_policies/ur3_pick_policy")
save_dir.mkdir(parents=True, exist_ok=True)

# Params - same flax serialization the cluster + deployment loader use.
with open(save_dir / "params.msgpack", "wb") as f:
    f.write(serialization.to_bytes(params))

metadata = {
    "policy_name": "ur3_pick_policy",
    "env_name": ENV_NAME,
    "obs_dim": int(env.observation_size),
    "action_dim": int(env.action_size),
    "network_factory": {
        k: (list(v) if isinstance(v, (tuple, list)) else v)
        for k, v in nf_params.items()
    },
    "notes": "UR3 + Hand-E pick policy trained locally with Brax PPO.",
}
with open(save_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", sorted(p.name for p in save_dir.iterdir()))
print(json.dumps(metadata, indent=2))